<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  15. Embeddings Project
</h2>

In this project, we will save various stock data sources to a ChromaDB vector database:
- **Earnings reports**: PDF -> OCR -> upload each page as a separate document to the vector database
- **Earnings calls**: YouTube video -> get transcript -> upload chunks of 20 entries as separate documents to the vector database

We will create an agent which can query the vector database for us.

In [ ]:
%pip install youtube-transcript-api langchain langchain-mistralai mistralai chromadb python-dotenv

In [ ]:
from mistralai.client import Mistral
from dotenv import load_dotenv
import os

load_dotenv()
mistral = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

### **15.1 ChromaDB Resources**

In [ ]:
import chromadb
from typing import Dict, Any
from chromadb import Documents, EmbeddingFunction, Embeddings
from chromadb.utils.embedding_functions import register_embedding_function

chroma_client = chromadb.PersistentClient(path="./chroma")

@register_embedding_function
class MistralEmbedding(EmbeddingFunction):

    def __init__(self, model: str = "mistral-embed"):
        self.model = model
        self.client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

    def __call__(self, input: Documents) -> Embeddings:
        response = self.client.embeddings.create(
            model=self.model,
            inputs=input
        )
        return [data.embedding for data in response.data]

    @staticmethod
    def name() -> str:
        return "mistral-embedding-function"

    def get_config(self) -> Dict[str, Any]:
        return dict(model=self.model)

    @staticmethod
    def build_from_config(config: Dict[str, Any]) -> "EmbeddingFunction":
        return MistralEmbedding(config['model'])

In [ ]:
collection = chroma_client.get_or_create_collection(
    name="earnings-collection",
    embedding_function=MistralEmbedding()
)

### **15.2 PDF Ingestion**

OCR tutorial series: https://www.youtube.com/watch?v=UcI3Ws1cK44&list=PLWDWab8QuiUk

Our PDF earnings reports contain relatively small amounts of text per page,<br>
allowing for a simplified chunking strategy where each page is indexed as an individual document in ChromaDB.

In [4]:
from pathlib import Path

files_to_ingest = list(Path("docs_for_embeddings").glob("*.pdf"))
files_to_ingest

[PosixPath('docs_for_embeddings/TSLA-Q1-2026-Update.pdf'),
 PosixPath('docs_for_embeddings/TSLA-Q4-2025-Update.pdf')]

In [6]:
import base64

def encode_file_to_base64(file_path: str) -> str:
    with open(file_path, "rb") as file:
        file_contents = file.read()
    return base64.b64encode(file_contents).decode("utf-8")

In [7]:
from mistralai.client.models import DocumentURLChunk
from mistralai.extra import response_format_from_pydantic_model
from pydantic import BaseModel, Field
from typing import Literal

class Illustration(BaseModel):
    type: Literal["chart", "image", "other"]
    description: str = Field(..., description="Description of the illustration, including key findings")

def ocr_pdf(file_path: str) -> tuple:

    pdf_base64 = encode_file_to_base64(file_path)
    document_url = f"data:application/pdf;base64,{pdf_base64}"

    ocr_response = mistral.ocr.process(
        model = "mistral-ocr-latest",
        document=DocumentURLChunk(document_url=document_url),
        bbox_annotation_format=response_format_from_pydantic_model(Illustration),
        pages="0-13"
    )

    return ocr_response, file_path.name

In [8]:
ocr_responses = []

for file_path in files_to_ingest:
    response = ocr_pdf(file_path=file_path)
    ocr_responses.append(response)

In [9]:
from mistralai.client.models import OCRResponse
import json

def upload_pdf_to_collection(ocr_response: OCRResponse, filename: str) -> bool:
    try:
        documents = []
        metadatas = []
        ids = []

        # Go through each page
        for page in ocr_response.pages:

            # Markdown
            page_text = page.markdown
            
            # Images
            image_descriptions = []

            for img in page.images:
                if img.image_annotation:
                    _annotations = json.loads(img.image_annotation)
                    _type = _annotations["type"]
                    _desc = _annotations["description"]
                    image_descriptions.append(f"[Image Context: {_type}: {_desc}]")
            
            if image_descriptions:
                page_text += "\n\n### Embedded Image Descriptions\n" + "\n".join(image_descriptions)

            # Skip empty pages
            if not page_text.strip():
                continue

            # Store full page as a single document
            page_num = int(page.index) + 1

            documents.append(page_text)
            metadatas.append({
                "source_type": "earnings_report",
                "source_name": filename,
                "page": page_num,
                "period": filename[5:12],
                "ticker": "TSLA"
            })
            ids.append(f"{filename}_p{page_num}")

        # Upload to ChromaDB
        if documents:
            collection.add(
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
            print(f"Successfully vectorized and loaded {len(documents)} pages from document {filename} into 'earnings_collection'.")
            return True
        
        return False
    
    except Exception as e:
        print(f"Error uploading {filename}: {e}")
        return False

In [10]:
for response in ocr_responses:
    upload_pdf_to_collection(
        ocr_response=response[0],
        filename=response[1]
    )

Successfully vectorized and loaded 14 pages from document TSLA-Q1-2026-Update.pdf into 'earnings_collection'.
Successfully vectorized and loaded 14 pages from document TSLA-Q4-2025-Update.pdf into 'earnings_collection'.


In [11]:
collection.query(
    query_texts="production figures",
    where={"period": "Q1-2026"},
    n_results=5
)

{'ids': [['TSLA-Q1-2026-Update.pdf_p6',
   'TSLA-Q1-2026-Update.pdf_p5',
   'TSLA-Q1-2026-Update.pdf_p2',
   'TSLA-Q1-2026-Update.pdf_p10',
   'TSLA-Q1-2026-Update.pdf_p4']],
 'embeddings': None,
 'documents': [["MANUFACTURING & HARDWARE\n\n# **Automotive**\n\nWe are focused on optimizing our vehicle product portfolio, with an emphasis on vehicles designed for a fully autonomous future. We continued the launch of Model 3 and Model Y trims globally, including the roll-out of the Model YL in markets outside of China and more affordable trims of both models. We also began deliveries of Cybertruck in the UAE.\n\nWe expect volume production of both Cybercab and the Tesla Semi this year.\n\n# **Energy generation and storage**\n\nProgress continued at the new Megafactory outside Houston, which will produce the Megapack 3 for Megablock. Start of production is on track for later this year. We began meaningful customer deployments of Tesla's first in-house designed solar panel produced at Gigafa

### **15.3 Transcripts Ingestion**

For simplicity, transcripts are chunked using a basic rule: a new chunk is created every 10 transcript entries or whenever a speaker transition occurs (indicated by >>).

A more robust production approach would incorporate **chunk overlap** (e.g., overlapping contiguous segments by 2–3 entries or a fixed token count)<br>
to preserve context across boundaries and prevent key information from being split mid-thought.

In [14]:
from youtube_transcript_api import YouTubeTranscriptApi
import uuid

ytt_api = YouTubeTranscriptApi()

def get_and_upload_transscript(video: dict, entries_per_chunk=20) -> tuple:

    fetched_transcript = ytt_api.fetch(video["id"], languages=['en'])
    transcript = fetched_transcript.to_raw_data()

    documents = []
    metadatas = []
    ids = []
    
    entries = []
    start_time = None
    
    for entry in transcript:
        
        entry_text = entry['text'].strip()
        
        if entry_text.lower() in ['[music]']:
            continue
            
        # Detect simple speaker transitions
        if entry_text.startswith(">>"):

            # If we already have a running chunk accumulated, save it before shifting speakers
            if entries:

                documents.append(" ".join(entries))

                metadatas.append({
                    "source_type": "earnings_call",
                    "source_name": f"YouTube video {video["id"]}: {video["name"]}",
                    "start_time": start_time,
                    "period": video["period"],
                    "ticker": "TSLA"
                })

                ids.append(str(uuid.uuid4()))

                entries = []
            
            entry_text = entry_text.lstrip("> ").strip()
            
        if not entries:
            start_time = entry['start']
            
        entries.append(entry_text)
        
        # Save document if the number of entries > entries_per_chunk
        if len(entries) >= entries_per_chunk:
            documents.append(" ".join(entries))
            metadatas.append({
                "source_type": "earnings_call",
                "source_name": f"YouTube video {video["id"]}: {video["name"]}",
                "start_time": start_time,
                "period": video["period"],
                "ticker": "TSLA"
            })
            ids.append(str(uuid.uuid4()))
            entries = []
            
    # Flush remaining segments
    if entries:
        documents.append(" ".join(entries))
        metadatas.append({
                "source_type": "earnings_call",
                "source_name": f"YouTube video {video["id"]}: {video["name"]}",
                "start_time": start_time,
                "period": video["period"],
                "ticker": "TSLA"
            })
        ids.append(str(uuid.uuid4()))
        
    return documents, metadatas, ids

In [15]:
videos = [
    {"id": "qO7T5zgRvXM", "name": "Tesla Q1 2026 Financial Results and Q&A Webcast", "period": "Q1-2026"},
    {"id": "oK0UZEE9GPo", "name": "Tesla Q4 and full year 2025 Financial Results and Q&A Webcast", "period": "Q4-2025"}
]

for video in videos:

    docs, meta, doc_ids = get_and_upload_transscript(video=video)

    collection.add(
        documents=docs,
        metadatas=meta,
        ids=doc_ids
    )

    print(f"Successfully indexed {len(docs)} chunks into ChromaDB.")

Successfully indexed 138 chunks into ChromaDB.
Successfully indexed 111 chunks into ChromaDB.


### **15.4 Agent**

LangChain tutorial series: https://www.youtube.com/watch?v=iUwaUVXfC_k&list=PLJu3ntxSDxFGS2xxwlMzr3qOaFotgj7OI

In [16]:
from langchain.tools import tool

@tool
def hybrid_search(
    query_texts: list,
    ids: list | None = None,
    metadata_filter_expression: dict | None = None,
    limit: int = 5
) -> dict:
    """Semantically searches Tesla earnings reports and call transcripts"""
    try:
        results = collection.query(
            ids=ids,
            query_texts=query_texts,
            where=metadata_filter_expression,
            n_results=limit
        )
        return results
    except Exception as e:
        return {"Error": str(e)}

In [17]:
system_prompt = """ 
You are a financial analyst assistant specializing in Tesla (TSLA) quarterly earnings and financial performance.
Your job is to answer user queries accurately by retrieving relevant information using the `hybrid_search` tool.

---

### DATA SOURCES & AVAILABLE METADATA FIELDS

The vector database contains two primary types of documents indexed with specific metadata fields:

1. **Earnings Reports (PDFs)**
   - `source_type`: `"earnings_report"`
   - `source_name`: Filename (e.g., `"TSLA-Q1-2026.pdf"`)
   - `ticker`: `"TSLA"`
   - `period`: Formatted as `"Q[1-4]-YYYY"` (e.g., `"Q1-2026"`, `"Q4-2025"`)
   - `page`: Page number (integer, starting at 1)

2. **Earnings Call Transcripts (YouTube Webcasts)**
   - `source_type`: `"earnings_call"`
   - `source_name`: String containing video title & ID
   - `ticker`: `"TSLA"`
   - `period`: Formatted as `"Q[1-4]-YYYY"` (e.g., `"Q1-2026"`, `"Q4-2025"`)
   - `start_time`: Video timestamp offset in seconds (float)

---

### TOOL USAGE INSTRUCTIONS (`hybrid_search`)

When invoking `hybrid_search`, populate the following parameters:
- `query_texts`: List of search strings (e.g., `["production numbers", "operating margin"]`).
- `metadata_filter_expression`: Dict containing a valid ChromaDB `where` filter (optional).
- `limit`: Number of results to retrieve (default: 5).

#### ChromaDB Filter Syntax Guide

To filter search results using `metadata_filter_expression`, strictly adhere to ChromaDB `where` clause rules:

1. **Exact Match / Shorthand Equality:**
   {"period": "Q1-2026"}

2. **Single Comparison Operators (`$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`):**
   {"period": {"$eq": "Q1-2026"}}
   {"source_type": {"$in": ["earnings_report", "earnings_call"]}}

3. **Multiple Conditions (Logical Operators `$and` / `$or`):**
   CRITICAL RULE: ChromaDB requires logical operators when combining multiple fields or conditions. You cannot pass multiple top-level keys in a single dictionary without `$and` or `$or`.

   - **Correct (Combining multiple conditions):**
     {
       "$and": [
         {"period": {"$eq": "Q1-2026"}},
         {"source_type": {"$eq": "earnings_report"}}
       ]
     }

   - **Incorrect (Will cause a ChromaDB error):**
     {"period": "Q1-2026", "source_type": "earnings_report"}

---

### EXAMPLE QUERY EXECUTION PATTERNS

- **User:** "What were Tesla's Q1 2026 production numbers in the earnings report?"
  - **Tool call:**
    hybrid_search(
        query_texts=["production and delivery figures"],
        metadata_filter_expression={
            "$and": [
                {"period": "Q1-2026"},
                {"source_type": "earnings_report"}
            ]
        },
        limit=5
    )

- **User:** "What did management discuss regarding AI or FSD during earnings calls?"
  - **Tool call:**
    hybrid_search(
        query_texts=["Full Self-Driving FSD progress artificial intelligence"],
        metadata_filter_expression={"source_type": "earnings_call"},
        limit=5
    )

---

### RESPONSE RULES
1. Always prefer using `hybrid_search` when specific financial metrics, quotes, or figures are requested.
2. If the user asks about a specific period (e.g., Q1 2026), always apply the appropriate `metadata_filter_expression` to narrow down the search.
3. Base your final answer strictly on the retrieved contexts. Always cite the period and document type (earnings report page or call transcript) in your summary.
"""

In [18]:
from langchain_mistralai import ChatMistralAI
from langchain.agents import create_agent

llm = ChatMistralAI(model_name="mistral-small-latest")
llm_with_tools = llm.bind_tools([hybrid_search])

# Add a checkpointer if you want to do more than a single turn conversation
agent = create_agent(
    model=llm_with_tools,
    tools=[hybrid_search],
    system_prompt=system_prompt
)

In [19]:
state = agent.invoke(
    {"messages": "How has the gross margin evolves from q4 2025 to q1 2026 according to the earnings report?"}
)

for msg in state["messages"]:
    msg.pretty_print()

================================ Human Message =================================

How has the gross margin evolves from q4 2025 to q1 2026 according to the earnings report?
================================== Ai Message ==================================
Tool Calls:
  hybrid_search (5dSo6SJdR)
 Call ID: 5dSo6SJdR
  Args:
    query_texts: ['gross margin evolution', 'gross margin trend', 'gross margin percentage']
    metadata_filter_expression: {'$and': [{'period': {'$in': ['Q4-2025', 'Q1-2026']}}, {'source_type': 'earnings_report'}]}
    limit: 10
================================= Tool Message =================================
Name: hybrid_search

{"ids": [["TSLA-Q4-2025-Update.pdf_p5", "TSLA-Q1-2026-Update.pdf_p4", "TSLA-Q4-2025-Update.pdf_p4", "TSLA-Q4-2025-Update.pdf_p7", "TSLA-Q4-2025-Update.pdf_p13", "TSLA-Q1-2026-Update.pdf_p10", "TSLA-Q4-2025-Update.pdf_p11", "TSLA-Q4-2025-Update.pdf_p6", "TSLA-Q1-2026-Update.pdf_p5", "TSLA-Q1-2026-Update.pdf_p8"], ["TSLA-Q4-2025-Update.pdf_p5", 

In [20]:
state = agent.invoke(
    {"messages": "What updates did Elon give regarding the Optimus production schedule in the Q1 2026 earnings call?"}
)

for msg in state["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What updates did Elon give regarding the Optimus production schedule in the Q1 2026 earnings call?
================================== Ai Message ==================================
Tool Calls:
  hybrid_search (8rWc6WVjy)
 Call ID: 8rWc6WVjy
  Args:
    query_texts: ['Optimus production schedule updates Elon Musk']
    metadata_filter_expression: {'$and': [{'period': 'Q1-2026'}, {'source_type': 'earnings_call'}]}
    limit: 5
================================= Tool Message =================================
Name: hybrid_search

{"ids": [["367f3876-5690-4787-a1c0-4545ed979590", "e57c84b5-0186-4bd8-ab44-b652431de800", "9661347e-a707-4e7e-a99b-1d5911de63ba", "b9f78707-be76-4b56-99e4-b2631616fb47", "0d645f70-717a-442c-8ee6-1f75afd0f75a"]], "embeddings": null, "documents": [["production in the future, and of course uh a very significant increase, well, uh actually releasing Optimus, um but increasing our internal 